# NLP Pipeline in PyTorch: Tokenization, Vocabulary, Embedding, and Classification

1. Install & Imports

In [ ]:
!pip install torch torchtext
import torch
import torch.nn as nn
from collections import Counter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.5 MB/s eta 0:00:00


2. Sample Data

In [ ]:
sentences = [
    "i love machine learning",
    "i love deep learning",
    "machine learning is amazing",
    "deep learning is powerful"
]

labels = torch.tensor([1, 1, 0, 0], dtype=torch.float32)

3. Tokenization

In [ ]:
def tokenize(sentence):
    return sentence.split()

tokenized_sentences = [tokenize(s) for s in sentences]

In [ ]:
tokenized_sentences[:2]

[['i', 'love', 'machine', 'learning'], ['i', 'love', 'deep', 'learning']]

4. Build Vocabulary

In [ ]:
word_counter = Counter()

for sentence in tokenized_sentences:
    word_counter.update(sentence)

vocab = {word: idx+1 for idx, (word, _) in enumerate(word_counter.items())}
vocab["<PAD>"] = 0

vocab_size = len(vocab)
print(vocab)
vocab_size

{'i': 1, 'love': 2, 'machine': 3, 'learning': 4, 'deep': 5, 'is': 6, 'amazing': 7, 'powerful': 8, '<PAD>': 0}


9

5. Encode Sentences

In [ ]:
def encode(sentence, vocab):
    return [vocab[word] for word in sentence]

encoded_sentences = [encode(s, vocab) for s in tokenized_sentences]

In [ ]:
encoded_sentences[:2]

[[1, 2, 3, 4], [1, 2, 5, 4]]

6. Padding

In [ ]:
max_len = 5

def pad(seq, max_len):
    return seq + [0]*(max_len - len(seq))

padded_sentences = [pad(seq, max_len) for seq in encoded_sentences]

X = torch.tensor(padded_sentences)
print(X)

tensor([[1, 2, 3, 4, 0],
        [1, 2, 5, 4, 0],
        [3, 4, 6, 7, 0],
        [5, 4, 6, 8, 0]])


In [ ]:
X[0]

tensor([1, 2, 3, 4, 0])

7. Embedding Layer

In [ ]:
embedding_dim = 8

embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

embedded = embedding(X)
print(embedded.shape)  # (batch_size, seq_len, embedding_dim)

torch.Size([4, 5, 8])


In [ ]:
embedded[3]

tensor([[-1.0257,  0.5619,  0.1765, -0.9391,  1.2864,  1.9136,  1.8105, -0.9687],
        [ 0.5600,  1.6691,  0.4763,  0.3890, -2.2972, -1.3992,  2.7216, -1.4666],
        [ 0.5840, -1.2479,  0.6275,  1.1248,  1.2650,  0.7524,  1.1214, -0.3202],
        [-0.5950,  0.7872, -0.2746,  1.1908,  0.4674, -0.6869,  0.7875, -0.3644],
        [ 0.3964,  0.0940,  1.0198, -1.7520, -0.5478,  0.3046,  1.6808,  0.0411]],
       grad_fn=<SelectBackward0>)

8. Model

In [ ]:
class SimpleNLPModel(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(embed_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)              # (B, L, E)
        x = x.permute(0, 2, 1)             # (B, E, L)
        x = self.pool(x).squeeze(-1)       # (B, E)
        x = self.fc(x)                     # (B, 1)
        return self.sigmoid(x)

In [ ]:
model = SimpleNLPModel(vocab_size, embedding_dim)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
epochs = 10

for epoch in range(epochs):
    optimizer.zero_grad()

    outputs = model(X).squeeze()
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.6413
Epoch 2, Loss: 0.6313
Epoch 3, Loss: 0.6214
Epoch 4, Loss: 0.6115
Epoch 5, Loss: 0.6017
Epoch 6, Loss: 0.5919
Epoch 7, Loss: 0.5822
Epoch 8, Loss: 0.5726
Epoch 9, Loss: 0.5629
Epoch 10, Loss: 0.5533


In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(X).squeeze()

    # Convert probabilities → predictions (0 or 1)
    preds = (outputs >= 0.5).int()

In [ ]:
correct = (preds == labels.int()).sum().item()
total = labels.size(0)

accuracy = correct / total

print(f"Accuracy: {accuracy*100:.4f}%")

Accuracy: 100.0000%


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(labels.cpu(), preds.cpu()))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00         2
         1.0       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4

